# Module 2: Exploratory Facial Motion Analysis

**Author:** Alex Argese

This notebook has two separate measurements: FLAME mesh displacements on one
VOCASET/FaceFormer pair, and MediaPipe landmark displacements on five animated clips.
Their regions, coordinate systems and tracking errors differ. The resulting ratios
are descriptive within each pipeline; they do not establish an eightfold difference
in expression demand or evaluate FaceFormer on cinematic audio.

**Revision:** historical outputs have been cleared because execution order, region
thresholds and temporal alignment were inconsistent. Rerun with the original inputs
and verified timing metadata before reporting new numerical results.

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print(f'numpy version: {np.__version__}')

---
## 2. Load Data

Upload the two `.npy` files manually using the Colab file uploader, then run this cell.

- `FaceTalk_170731_00024_TA_sentence21.npy` → ground truth
- `FaceTalk_170731_00024_TA_sentence21_condition_FaceTalk_170725_00137_TA.npy` → FaceFormer prediction

Both files are available at:  
https://github.com/kaist-ami/Perceptual-3D-Talking-Head/tree/main/evaluation/data_MTM

> **Note:** the files are stored in Git LFS in the source repository. Download them directly from the GitHub page linked above and upload to your Colab session.

In [ ]:
GT_PATH   = 'FaceTalk_170731_00024_TA_sentence21.npy'
PRED_PATH = 'FaceTalk_170731_00024_TA_sentence21_condition_FaceTalk_170725_00137_TA.npy'

gt   = np.load(GT_PATH,   allow_pickle=False)
pred = np.load(PRED_PATH, allow_pickle=False)

print(f'Ground truth shape:          {gt.shape}   dtype: {gt.dtype}')
print(f'FaceFormer prediction shape: {pred.shape}  dtype: {pred.dtype}')

## 3. Validate timing and compute displacement

The original files contain 278 GT frames and 138 prediction frames. Frame counts
alone do not identify their sampling rates or synchronization. Supply verified FPS
and confirm a shared start time; do not infer FPS from the length ratio. The cell
below resamples GT to prediction timestamps over their shared support and rejects
material duration mismatches. Neither sequence's first frame is guaranteed neutral.
The statistic measures displacement relative to each sequence's own first frame,
not velocity, reconstruction error, or perceptual expression quality.

In [ ]:
def align_mesh_sequences(gt, pred, gt_fps, pred_fps, same_start_confirmed=False):
    if not same_start_confirmed:
        raise ValueError('Verify the same utterance and synchronized start times first.')
    for name, fps in [('GT', gt_fps), ('prediction', pred_fps)]:
        if fps is None or not np.isfinite(fps) or fps <= 0:
            raise ValueError(f'Provide verified positive {name} FPS; do not infer it from frame counts.')
    arrays = []
    for name, values in [('GT', gt), ('prediction', pred)]:
        values = np.asarray(values)
        if values.ndim == 2 and values.shape[1] == 5023 * 3:
            values = values.reshape(-1, 5023, 3)
        if values.ndim != 3 or values.shape[1:] != (5023, 3) or len(values) < 2:
            raise ValueError(f'{name} must contain at least two FLAME frames with 5023 vertices.')
        if not np.isfinite(values).all():
            raise ValueError(f'{name} contains non-finite coordinates.')
        arrays.append(values)
    gt_v, pred_v = arrays
    gt_t = np.arange(len(gt_v)) / gt_fps
    pred_t = np.arange(len(pred_v)) / pred_fps
    tolerance = 2 * max(1 / gt_fps, 1 / pred_fps)
    if abs(gt_t[-1] - pred_t[-1]) > tolerance:
        raise ValueError('Duration mismatch: verify FPS, utterance identity and temporal offsets.')
    keep = pred_t <= gt_t[-1]
    times = pred_t[keep]
    if len(times) < 2:
        raise ValueError('Insufficient common temporal support.')
    right = np.clip(np.searchsorted(gt_t, times, side='right'), 1, len(gt_t) - 1)
    left = right - 1
    weight = ((times - gt_t[left]) / (gt_t[right] - gt_t[left]))[:, None, None]
    aligned_gt = gt_v[left] * (1 - weight) + gt_v[right] * weight
    return aligned_gt, pred_v[keep], times

# Fill these from the source data/preprocessing documentation before proceeding.
GT_FPS = None
PRED_FPS = None
SAME_START_CONFIRMED = False
gt_v, pred_v, mesh_times = align_mesh_sequences(
    gt, pred, GT_FPS, PRED_FPS, SAME_START_CONFIRMED
)
n_frames = len(mesh_times)
gt_disp = np.linalg.norm(gt_v - gt_v[0], axis=2) * 1000
pred_disp = np.linalg.norm(pred_v - pred_v[0], axis=2) * 1000
print(f'Aligned frames: {n_frames}; last timestamp: {mesh_times[-1]:.3f} s')

## 4. Geometric facial regions (FLAME only)

Coordinates must use the original orientation and meters. The lip box is
`abs(x) < .035`, `-.075 < y < -.025`, `z > -.04`.
The upper-face box is `y > .015`, `abs(x) < .075`, `z > -.06`.
This matches the executable notebook, whose stored output reported 405 and 1766
vertices. The old paper instead described `.005` and 2300 vertices. Counts are
computed below, not assumed. These boxes are coordinate-dependent anatomical
proxies; they are not invariant to mesh pose or scale.

In [ ]:
import plotly.graph_objects as go

vertices = gt_v[0]

x = vertices[:, 0]
y = vertices[:, 1]
z = vertices[:, 2]

print("x range:", x.min(), x.max())
print("y range:", y.min(), y.max())
print("z range:", z.min(), z.max())

In [ ]:
# Regions defined by coordinate thresholds on the neutral pose
# Verified visually using the 3D scatter plot above

lip_mask = (
    (np.abs(x) < 0.035) &   # center face
    (y < -0.025) &          # low side
    (y > -0.075) &          # not too much
    (z > -0.04)             # frontal part
)

upper_mask = (
    (y > 0.015) &           # upper side
    (np.abs(x) < 0.075) &   # avoid ears
    (z > -0.06)             # frontal part
)

flame_lip_idx = np.where(lip_mask)[0]
flame_upper_idx = np.where(upper_mask)[0]

print("Lip vertices:", len(flame_lip_idx))
print("Upper-face vertices:", len(flame_upper_idx))
if len(flame_lip_idx) == 0 or len(flame_upper_idx) == 0:
    raise ValueError('Empty region: check coordinate orientation and units.')

In [ ]:
def visualize_regions(vertices, flame_lip_idx, flame_upper_idx):
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        mode="markers",
        marker=dict(size=1, opacity=0.08),
        name="All vertices"
    ))

    fig.add_trace(go.Scatter3d(
        x=vertices[flame_lip_idx, 0],
        y=vertices[flame_lip_idx, 1],
        z=vertices[flame_lip_idx, 2],
        mode="markers",
        marker=dict(size=5),
        name="Lip region"
    ))

    fig.add_trace(go.Scatter3d(
        x=vertices[flame_upper_idx, 0],
        y=vertices[flame_upper_idx, 1],
        z=vertices[flame_upper_idx, 2],
        mode="markers",
        marker=dict(size=4),
        name="Upper-face region"
    ))

    fig.update_layout(
        title="Geometric lip and upper-face regions",
        scene=dict(aspectmode="data"),
        width=900,
        height=750
    )

    fig.show()

visualize_regions(vertices, flame_lip_idx, flame_upper_idx)

---
## 5. Results: Ground Truth vs FaceFormer

Two plots:
1. **Bar chart**: mean displacement per facial region, GT vs FaceFormer
2. **Time series**: lip vs upper-face motion over time, GT vs FaceFormer

In [ ]:
import pandas as pd

# Sequences were aligned by timestamp in section 3.
gt_motion = gt_v - gt_v[0:1]
pred_motion = pred_v - pred_v[0:1]
gt_lip_ts = np.linalg.norm(gt_motion[:, flame_lip_idx, :], axis=2).mean(axis=1) * 1000
pred_lip_ts = np.linalg.norm(pred_motion[:, flame_lip_idx, :], axis=2).mean(axis=1) * 1000
gt_upper_ts = np.linalg.norm(gt_motion[:, flame_upper_idx, :], axis=2).mean(axis=1) * 1000
pred_upper_ts = np.linalg.norm(pred_motion[:, flame_upper_idx, :], axis=2).mean(axis=1) * 1000
summary = pd.DataFrame({
    'Region': ['Lips', 'Upper-face'],
    'GT': [gt_lip_ts.mean(), gt_upper_ts.mean()],
    'FaceFormer': [pred_lip_ts.mean(), pred_upper_ts.mean()],
})
summary

Interpret the recomputed table for this sequence only. Historical percentages
(-35% lips, -59% upper face) are not validated by this revision. The old notebook
contained inconsistent upper-face values and compared frame indices without
verified timing. Displacement amplitude does not establish animation accuracy.
LVE is an evaluation metric; the official FaceFormer implementation trains with
vertex MSE, so the old attribution to a lip-only training objective was incorrect.


### 5.1 Mean displacement per region

In [ ]:
ax = summary.plot(
    x="Region",
    y=["GT", "FaceFormer"],
    kind="bar",
    figsize=(7, 4),
    rot=0
)

ax.set_title("Mean Displacement per Facial Region")
ax.set_ylabel("Mean displacement (mm)")
ax.set_xlabel("")
ax.legend(title="")
plt.tight_layout()
plt.show()

## Time series — lip vs upper-face motion over time

In [ ]:
time = mesh_times

plt.figure(figsize=(10, 5))

plt.plot(time, gt_lip_ts, label="GT lips")
plt.plot(time, pred_lip_ts, label="FaceFormer lips")
plt.plot(time, gt_upper_ts, label="GT upper-face")
plt.plot(time, pred_upper_ts, label="FaceFormer upper-face")

plt.title("Regional Facial Motion Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Mean displacement (mm)")
plt.legend()
plt.tight_layout()
plt.show()

# DCC landmark extraction (separate pipeline)

Requires `videos/hans.mp4`, `gothel.mp4`, `miguel.mp4`, `dolores.mp4`, `moana.mp4`
and the MediaPipe face landmarker model. These inputs are not distributed here.
Record source edition, scene timecodes, crop/shot boundaries, FPS and SHA-256 for
each clip before a reproducible evaluation. Check tracking on animated characters.
Nose centering and face-width scaling do not remove head rotation, perspective,
camera movement or tracking error. MediaPipe coordinates are not metric FLAME XYZ.

In [ ]:
%pip install mediapipe opencv-python

In [ ]:
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

In [ ]:
import cv2
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


model_path = "face_landmarker.task"

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)


def extract_landmarks_new(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')
    landmarks_all = []
    frame_ids = []

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        cap.release()
        raise ValueError('Missing valid video FPS; do not use an arbitrary fallback.')

    with FaceLandmarker.create_from_options(options) as landmarker:

        frame_id = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb
            )

            timestamp_ms = int(frame_id * 1000 / fps)

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.face_landmarks:
                pts = np.array([[lm.x, lm.y, lm.z] for lm in result.face_landmarks[0]])
                landmarks_all.append(pts)
                frame_ids.append(frame_id)

            frame_id += 1

    cap.release()

    if not landmarks_all:
        raise ValueError(f'No face landmarks detected in {video_path}')
    return np.array(landmarks_all), np.array(frame_ids)

In [ ]:
# Landmark regions
mp_lip_idx = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    291, 308, 324, 318, 402, 317, 14, 87, 178, 88,
    95, 185, 40, 39, 37, 0, 267, 269, 270, 409,
    415, 310, 311, 312, 13, 82, 81, 42, 183, 78
]

mp_upper_idx = [
    70, 63, 105, 66, 107,
    336, 296, 334, 293, 300,
    33, 160, 158, 133, 153, 144,
    362, 385, 387, 263, 373, 380,
    10, 151, 9
]

def normalize_landmarks(lm):
    lm = np.asarray(lm, dtype=float)
    if lm.ndim != 3 or lm.shape[1:] != (478, 3) or len(lm) < 2:
        raise ValueError('Expected at least two frames of 478 MediaPipe landmarks.')
    if not np.isfinite(lm).all():
        raise ValueError('Landmarks contain non-finite values.')
    scale = np.linalg.norm(lm[:, 234] - lm[:, 454], axis=1)
    if np.any(scale <= 1e-8):
        raise ValueError('Degenerate face width; inspect tracking.')
    return (lm - lm[:, 1:2, :]) / scale[:, None, None]

def motion_timeseries(lm_norm, idx):
    idx = np.asarray(idx)
    if idx.ndim != 1 or idx.size == 0 or not np.issubdtype(idx.dtype, np.integer):
        raise ValueError('Region must contain integer landmark indices.')
    if np.any(idx < 0) or np.any(idx >= lm_norm.shape[1]):
        raise ValueError('Region indices do not match the landmark topology.')
    motion = lm_norm - lm_norm[0:1]
    return np.linalg.norm(motion[:, idx, :], axis=2).mean(axis=1)

In [ ]:
def show_landmarks_on_frame(video_path, frame_number, landmarks, frame_ids, mp_lip_idx, mp_upper_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ret, frame = cap.read()
    cap.release()

    if not ret:
        print("Frame non letto.")
        return

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame_rgb.shape

    nearest_i = np.argmin(np.abs(frame_ids - frame_number))
    pts = landmarks[nearest_i]

    plt.figure(figsize=(10, 7))
    plt.imshow(frame_rgb)

    plt.scatter(pts[:, 0] * w, pts[:, 1] * h, s=4, alpha=0.25, label="All landmarks")
    plt.scatter(pts[mp_lip_idx, 0] * w, pts[mp_lip_idx, 1] * h, s=25, label="Lips")
    plt.scatter(pts[mp_upper_idx, 0] * w, pts[mp_upper_idx, 1] * h, s=25, label="Upper-face")

    plt.title(f"Landmark verification — frame {frame_number}")
    plt.axis("off")
    plt.legend()
    plt.show()

## Frozen - Hans

In [ ]:
lm, frame_ids = extract_landmarks_new("videos/hans.mp4")

print("Landmarks shape:", lm.shape)

In [ ]:
lm_norm = normalize_landmarks(lm)

dcc_lip_ts = motion_timeseries(lm_norm, mp_lip_idx)
dcc_upper_ts = motion_timeseries(lm_norm, mp_upper_idx)

print("Hans DCC metrics")
print("Lip mean:", dcc_lip_ts.mean())
print("Lip p95:", np.percentile(dcc_lip_ts, 95))
print("Upper mean:", dcc_upper_ts.mean())
print("Upper p95:", np.percentile(dcc_upper_ts, 95))
print("Upper std:", dcc_upper_ts.std())
print("Upper/Lip ratio:", dcc_upper_ts.mean() / (dcc_lip_ts.mean() + 1e-8))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(frame_ids, dcc_lip_ts, label="DCC lips")
plt.plot(frame_ids, dcc_upper_ts, label="DCC upper-face")

plt.title("DCC Facial Motion Over Time — Hans Clip")
plt.xlabel("Frame")
plt.ylabel("Normalized displacement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
show_landmarks_on_frame("videos/hans.mp4", 10, lm, frame_ids, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/hans.mp4", 40, lm, frame_ids, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/hans.mp4", 72, lm, frame_ids, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/hans.mp4", 95, lm, frame_ids, mp_lip_idx, mp_upper_idx)

## Tangled - Gothel

In [ ]:
lm2, frame_ids2 = extract_landmarks_new("videos/gothel.mp4")

print("Landmarks shape:", lm2.shape)

In [ ]:
lm_norm2 = normalize_landmarks(lm2)

dcc_lip_ts2 = motion_timeseries(lm_norm2, mp_lip_idx)
dcc_upper_ts2 = motion_timeseries(lm_norm2, mp_upper_idx)

print("Gothel DCC metrics")
print("Lip mean:", dcc_lip_ts2.mean())
print("Lip p95:", np.percentile(dcc_lip_ts2, 95))
print("Upper mean:", dcc_upper_ts2.mean())
print("Upper p95:", np.percentile(dcc_upper_ts2, 95))
print("Upper std:", dcc_upper_ts2.std())
print("Upper/Lip ratio:", dcc_upper_ts2.mean() / (dcc_lip_ts2.mean() + 1e-8))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(frame_ids2, dcc_lip_ts2, label="DCC lips")
plt.plot(frame_ids2, dcc_upper_ts2, label="DCC upper-face")

plt.title("DCC Facial Motion Over Time — Gothel Clip")
plt.xlabel("Frame")
plt.ylabel("Normalized displacement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
show_landmarks_on_frame("videos/gothel.mp4", 1, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/gothel.mp4", 10, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/gothel.mp4", 52, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/gothel.mp4", 58, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/gothel.mp4", 120, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/gothel.mp4", 147, lm2, frame_ids2, mp_lip_idx, mp_upper_idx)

## Coco - Miguel

In [ ]:
lm3, frame_ids3 = extract_landmarks_new("videos/miguel.mp4")

print("Landmarks shape:", lm3.shape)

In [ ]:
lm_norm3 = normalize_landmarks(lm3)

dcc_lip_ts3 = motion_timeseries(lm_norm3, mp_lip_idx)
dcc_upper_ts3 = motion_timeseries(lm_norm3, mp_upper_idx)

print("Miguel DCC metrics")
print("Lip mean:", dcc_lip_ts3.mean())
print("Lip p95:", np.percentile(dcc_lip_ts3, 95))
print("Upper mean:", dcc_upper_ts3.mean())
print("Upper p95:", np.percentile(dcc_upper_ts3, 95))
print("Upper std:", dcc_upper_ts3.std())
print("Upper/Lip ratio:", dcc_upper_ts3.mean() / (dcc_lip_ts3.mean() + 1e-8))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(frame_ids3, dcc_lip_ts3, label="DCC lips")
plt.plot(frame_ids3, dcc_upper_ts3, label="DCC upper-face")

plt.title("DCC Facial Motion Over Time — Miguel Clip")
plt.xlabel("Frame")
plt.ylabel("Normalized displacement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
show_landmarks_on_frame("videos/miguel.mp4", 1, lm3, frame_ids3, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/miguel.mp4", 20, lm3, frame_ids3, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/miguel.mp4", 32, lm3, frame_ids3, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/miguel.mp4", 58, lm3, frame_ids3, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/miguel.mp4", 67, lm3, frame_ids3, mp_lip_idx, mp_upper_idx)

## Encanto - Dolores

In [ ]:
lm4, frame_ids4 = extract_landmarks_new("videos/dolores.mp4")

print("Landmarks shape:", lm4.shape)

In [ ]:
lm_norm4 = normalize_landmarks(lm4)

dcc_lip_ts4 = motion_timeseries(lm_norm4, mp_lip_idx)
dcc_upper_ts4 = motion_timeseries(lm_norm4, mp_upper_idx)

print("Dolores DCC metrics")
print("Lip mean:", dcc_lip_ts4.mean())
print("Lip p95:", np.percentile(dcc_lip_ts4, 95))
print("Upper mean:", dcc_upper_ts4.mean())
print("Upper p95:", np.percentile(dcc_upper_ts4, 95))
print("Upper std:", dcc_upper_ts4.std())
print("Upper/Lip ratio:", dcc_upper_ts4.mean() / (dcc_lip_ts4.mean() + 1e-8))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(frame_ids4, dcc_lip_ts4, label="DCC lips")
plt.plot(frame_ids4, dcc_upper_ts4, label="DCC upper-face")

plt.title("DCC Facial Motion Over Time — Dolores Clip")
plt.xlabel("Frame")
plt.ylabel("Normalized displacement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
show_landmarks_on_frame("videos/dolores.mp4", 1, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/dolores.mp4", 20, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/dolores.mp4", 50, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/dolores.mp4", 90, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/dolores.mp4", 150, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/dolores.mp4", 200, lm4, frame_ids4, mp_lip_idx, mp_upper_idx)

## Moana - Moana

In [ ]:
lm5, frame_ids5 = extract_landmarks_new("videos/moana.mp4")

print("Landmarks shape:", lm5.shape)

In [ ]:
lm_norm5 = normalize_landmarks(lm5)

dcc_lip_ts5 = motion_timeseries(lm_norm5, mp_lip_idx)
dcc_upper_ts5 = motion_timeseries(lm_norm5, mp_upper_idx)

print("Moana DCC metrics")
print("Lip mean:", dcc_lip_ts5.mean())
print("Lip p95:", np.percentile(dcc_lip_ts5, 95))
print("Upper mean:", dcc_upper_ts5.mean())
print("Upper p95:", np.percentile(dcc_upper_ts5, 95))
print("Upper std:", dcc_upper_ts5.std())
print("Upper/Lip ratio:", dcc_upper_ts5.mean() / (dcc_lip_ts5.mean() + 1e-8))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(frame_ids5, dcc_lip_ts5, label="DCC lips")
plt.plot(frame_ids5, dcc_upper_ts5, label="DCC upper-face")

plt.title("DCC Facial Motion Over Time — Moana Clip")
plt.xlabel("Frame")
plt.ylabel("Normalized displacement")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
show_landmarks_on_frame("videos/moana.mp4", 1, lm5, frame_ids5, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/moana.mp4", 40, lm5, frame_ids5, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/moana.mp4", 95, lm5, frame_ids5, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/moana.mp4", 105, lm5, frame_ids5, mp_lip_idx, mp_upper_idx)
show_landmarks_on_frame("videos/moana.mp4", 120, lm5, frame_ids5, mp_lip_idx, mp_upper_idx)

# DCC analysis

In [ ]:
import numpy as np
import pandas as pd

dcc_clips = {
    "hans": (lm, frame_ids),
    "gothel": (lm2, frame_ids2),
    "miguel": (lm3, frame_ids3),
    "dolores": (lm4, frame_ids4),
    "moana": (lm5, frame_ids5),
}

rows = []

for name, (landmarks, frames) in dcc_clips.items():
    lm_norm = normalize_landmarks(landmarks)

    lip_ts = motion_timeseries(lm_norm, mp_lip_idx)
    upper_ts = motion_timeseries(lm_norm, mp_upper_idx)

    rows.append({
        "clip": name,
        "n_frames": len(frames),
        "lip_mean": lip_ts.mean(),
        "lip_p95": np.percentile(lip_ts, 95),
        "lip_std": lip_ts.std(),
        "upper_mean": upper_ts.mean(),
        "upper_p95": np.percentile(upper_ts, 95),
        "upper_std": upper_ts.std(),
        "upper_lip_ratio": upper_ts.mean() / (lip_ts.mean() + 1e-8),
    })

dcc_df = pd.DataFrame(rows)
dcc_df

In [ ]:
dcc_df.to_csv("dcc_motion_metrics.csv", index=False)
print("Saved: dcc_motion_metrics.csv")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(dcc_df["clip"], dcc_df["upper_mean"])
plt.title("DCC Upper-Face Motion by Clip")
plt.xlabel("Clip")
plt.ylabel("Normalized upper-face motion")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# VOCASET vs. DCC

### Cross-pipeline comparison is not a calibrated effect size

FLAME mesh regions and MediaPipe regions contain different points and use different
coordinate systems and normalization. Dividing two dimensionless ratios does not
remove these differences. The old 0.15 vs 1.21 chart is retained only in the Git
history; it is not a validated eightfold increase in cinematic expression demand.
A future comparison needs matched regions, head-pose handling, tracking validation
and a common measurement pipeline on both corpora.

## Summary and next experiments

- Report the aligned FLAME results for this one sequence, with exact region counts.
- Report DCC tracking statistics separately; assess head rotation, shot cuts and failures.
- Use a common validated representation before estimating a cross-corpus motion gap.
- Evaluate multiple sequences and speakers; report sequence-level variability.
- Run FaceFormer on cinematic audio to test its behavior on that input domain.
- Test training-data and objective hypotheses with controlled ablations.

This revision corrects the analysis protocol; it does not claim new empirical results.